# 형태소 통계 분석

**작성일**: 2026-02-10  
**목적**: ㄴ 삽입 후보에서 반복되는 형태소 파악

---

## 📚 분석 내용

1. **빈도 높은 형태소**: 어떤 형태소가 자주 나오는지
2. **위치 분석**: 앞(morph1)에 주로 오는지, 뒤(morph2)에 주로 오는지
3. **유형 분석**: 접사인지 어근인지
4. **공기 패턴**: 특정 형태소 조합이 많은지
5. **ㄴ 삽입 비율**: 특정 형태소가 ㄴ 삽입을 촉진하는가? ⭐

---

## 1️⃣ 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

# CSV 파일 경로 (32_n_insertion.ipynb 실행 결과)
# 파일명 수정 필요!
csv_path = 'search_results/n_insertion_candidates_YYYYMMDD_HHMMSS_reviewed.csv'

df = pd.read_csv(csv_path, encoding='utf-8-sig')

print(f"전체 데이터: {len(df):,}개")
print(f"decision = keep: {(df['decision'] == 'keep').sum():,}개")

In [ ]:
# decision = keep인 것만 분석
df_keep = df[df['decision'] == 'keep'].copy()

print(f"분석 대상: {len(df_keep):,}개")

---

## 2️⃣ morph1 (선행 형태소) 분석

In [ ]:
# morph1 빈도
morph1_counts = df_keep['morph1'].value_counts()

print("=" * 60)
print("선행 형태소 (morph1) Top 20")
print("=" * 60)

for i, (morph, count) in enumerate(morph1_counts.head(20).items(), 1):
    # 해당 형태소의 예시
    examples = df_keep[df_keep['morph1'] == morph]['word'].head(3).tolist()
    examples_str = ', '.join(examples)
    
    print(f"{i:2d}. {morph:10s} ({count:3d}회) - 예: {examples_str}")

---

## 3️⃣ morph2 (후행 형태소) 분석

In [ ]:
# morph2 빈도
morph2_counts = df_keep['morph2'].value_counts()

print("=" * 60)
print("후행 형태소 (morph2) Top 20")
print("=" * 60)

for i, (morph, count) in enumerate(morph2_counts.head(20).items(), 1):
    # 해당 형태소의 예시
    examples = df_keep[df_keep['morph2'] == morph]['word'].head(3).tolist()
    examples_str = ', '.join(examples)
    
    print(f"{i:2d}. {morph:10s} ({count:3d}회) - 예: {examples_str}")

---

## 4️⃣ 형태소 위치 분석

In [ ]:
# 모든 형태소 수집
all_morphs = set(df_keep['morph1'].unique()) | set(df_keep['morph2'].unique())

print(f"고유 형태소: {len(all_morphs):,}개\n")

# 각 형태소가 앞/뒤에 몇 번씩 나오는지
morph_positions = []

for morph in all_morphs:
    count_morph1 = (df_keep['morph1'] == morph).sum()
    count_morph2 = (df_keep['morph2'] == morph).sum()
    total = count_morph1 + count_morph2
    
    if total >= 2:  # 2회 이상만
        morph_positions.append({
            'morph': morph,
            'as_morph1': count_morph1,
            'as_morph2': count_morph2,
            'total': total,
            'ratio_morph1': count_morph1 / total if total > 0 else 0
        })

df_positions = pd.DataFrame(morph_positions).sort_values('total', ascending=False)

print("=" * 70)
print("형태소 위치 분석 (Top 20)")
print("=" * 70)
print(f"{'형태소':10s} {'앞(morph1)':>12s} {'뒤(morph2)':>12s} {'합계':>8s} {'선호':>10s}")
print("-" * 70)

for _, row in df_positions.head(20).iterrows():
    prefer = "앞" if row['ratio_morph1'] > 0.7 else ("뒤" if row['ratio_morph1'] < 0.3 else "양쪽")
    print(f"{row['morph']:10s} {row['as_morph1']:12d} {row['as_morph2']:12d} {row['total']:8d} {prefer:>10s}")

---

## 5️⃣ 자주 나오는 조합 (공기 패턴)

In [ ]:
# morph1 + morph2 조합
combinations = df_keep.groupby(['morph1', 'morph2']).size().reset_index(name='count')
combinations = combinations.sort_values('count', ascending=False)

print("=" * 60)
print("자주 나오는 형태소 조합 Top 20")
print("=" * 60)
print(f"{'morph1':15s} + {'morph2':15s} {'빈도':>6s} {'예시':20s}")
print("-" * 60)

for _, row in combinations.head(20).iterrows():
    m1, m2, cnt = row['morph1'], row['morph2'], row['count']
    
    # 예시 단어
    example = df_keep[(df_keep['morph1'] == m1) & (df_keep['morph2'] == m2)]['word'].iloc[0]
    
    print(f"{m1:15s} + {m2:15s} {cnt:6d} {example:20s}")

---

## 6️⃣ 접사 vs 어근 분석 (seg_links 활용)

In [ ]:
# seg_links 예시:
# "SOM/NNG(537178:002)+I-BUl/NNG(1409052:008)"
# NNG = 일반명사, XPN = 접두사, XSN = 명사파생 접미사 등

def extract_pos_from_links(links_str):
    """
    seg_links에서 품사 추출
    예: "SOM/NNG+I-BUl/NNG" → ["NNG", "NNG"]
    """
    if not links_str or pd.isna(links_str):
        return []
    
    import re
    # /품사( 패턴 찾기
    pos_list = re.findall(r'/([A-Z]+)', links_str)
    return pos_list

# 품사 정보 추가
df_keep['pos_list'] = df_keep['seg_links'].apply(extract_pos_from_links)
df_keep['morph1_pos'] = df_keep['pos_list'].apply(lambda x: x[0] if len(x) >= 1 else '')
df_keep['morph2_pos'] = df_keep['pos_list'].apply(lambda x: x[1] if len(x) >= 2 else '')

print("=" * 60)
print("형태소 품사 분포")
print("=" * 60)

print("\nmorph1 (선행) 품사:")
print(df_keep['morph1_pos'].value_counts().head(10))

print("\nmorph2 (후행) 품사:")
print(df_keep['morph2_pos'].value_counts().head(10))

print("\n품사 코드:")
print("  NNG: 일반명사")
print("  NNP: 고유명사")
print("  XPN: 접두사")
print("  XSN: 명사파생 접미사")
print("  XR: 어근")
print("  MAG: 부사")

---

## 🎯 7️⃣ 형태소별 ㄴ 삽입 비율 분석 (핵심!)

**연구 질문**: 특정 형태소가 포함되면 ㄴ 삽입이 더 잘 일어나는가?

In [ ]:
# morph1별 ㄴ 삽입 비율
print("=" * 80)
print("선행 형태소 (morph1)별 ㄴ 삽입 비율 - Top 20")
print("=" * 80)
print(f"{'형태소':10s} {'총 출현':>8s} {'yes':>6s} {'no':>6s} {'maybe':>6s} {'dialect':>8s} {'yes 비율':>10s}")
print("-" * 80)

morph1_n_insertion = []

for morph in morph1_counts.head(20).index:
    subset = df_keep[df_keep['morph1'] == morph]
    total = len(subset)
    
    n_yes = (subset['n_insertion'] == 'yes').sum()
    n_no = (subset['n_insertion'] == 'no').sum()
    n_maybe = (subset['n_insertion'] == 'maybe').sum()
    n_dialect = (subset['n_insertion'] == 'dialect').sum()
    
    yes_ratio = n_yes / total if total > 0 else 0
    
    morph1_n_insertion.append({
        'morph': morph,
        'total': total,
        'n_yes': n_yes,
        'n_no': n_no,
        'n_maybe': n_maybe,
        'n_dialect': n_dialect,
        'yes_ratio': yes_ratio
    })
    
    print(f"{morph:10s} {total:8d} {n_yes:6d} {n_no:6d} {n_maybe:6d} {n_dialect:8d} {yes_ratio:9.1%}")

df_morph1_n_ins = pd.DataFrame(morph1_n_insertion)

In [ ]:
# morph2별 ㄴ 삽입 비율
print("\n" + "=" * 80)
print("후행 형태소 (morph2)별 ㄴ 삽입 비율 - Top 20")
print("=" * 80)
print(f"{'형태소':10s} {'총 출현':>8s} {'yes':>6s} {'no':>6s} {'maybe':>6s} {'dialect':>8s} {'yes 비율':>10s}")
print("-" * 80)

morph2_n_insertion = []

for morph in morph2_counts.head(20).index:
    subset = df_keep[df_keep['morph2'] == morph]
    total = len(subset)
    
    n_yes = (subset['n_insertion'] == 'yes').sum()
    n_no = (subset['n_insertion'] == 'no').sum()
    n_maybe = (subset['n_insertion'] == 'maybe').sum()
    n_dialect = (subset['n_insertion'] == 'dialect').sum()
    
    yes_ratio = n_yes / total if total > 0 else 0
    
    morph2_n_insertion.append({
        'morph': morph,
        'total': total,
        'n_yes': n_yes,
        'n_no': n_no,
        'n_maybe': n_maybe,
        'n_dialect': n_dialect,
        'yes_ratio': yes_ratio
    })
    
    print(f"{morph:10s} {total:8d} {n_yes:6d} {n_no:6d} {n_maybe:6d} {n_dialect:8d} {yes_ratio:9.1%}")

df_morph2_n_ins = pd.DataFrame(morph2_n_insertion)

---

## 📊 8️⃣ 사전 발음 vs 수동 검토 일치도

In [ ]:
# n_in_pron (사전) vs n_insertion (수동 검토) 비교
print("=" * 60)
print("사전 발음 (n_in_pron) vs 수동 검토 (n_insertion) 일치도")
print("=" * 60)

# n_insertion을 yes/no로 단순화
df_keep['n_insertion_binary'] = df_keep['n_insertion'].apply(
    lambda x: 'yes' if x in ['yes', 'dialect'] else ('no' if x == 'no' else 'uncertain')
)

# 교차표
crosstab = pd.crosstab(
    df_keep['n_in_pron'],
    df_keep['n_insertion_binary'],
    margins=True
)

print(crosstab)

# 일치율
matched = (
    ((df_keep['n_in_pron'] == 'yes') & (df_keep['n_insertion_binary'] == 'yes')).sum() +
    ((df_keep['n_in_pron'] == 'no') & (df_keep['n_insertion_binary'] == 'no')).sum()
)
total_certain = (df_keep['n_insertion_binary'] != 'uncertain').sum()
accuracy = matched / total_certain if total_certain > 0 else 0

print(f"\n일치율: {accuracy:.1%} (uncertain 제외)")

# 불일치 케이스 분석
print("\n불일치 케이스 (사전: yes, 실제: no):")
mismatch1 = df_keep[(df_keep['n_in_pron'] == 'yes') & (df_keep['n_insertion_binary'] == 'no')]
if len(mismatch1) > 0:
    print(mismatch1[['word', 'seg_morph', 'pron', 'notes']].head(10))
else:
    print("  없음")

print("\n불일치 케이스 (사전: no, 실제: yes):")
mismatch2 = df_keep[(df_keep['n_in_pron'] == 'no') & (df_keep['n_insertion_binary'] == 'yes')]
if len(mismatch2) > 0:
    print(mismatch2[['word', 'seg_morph', 'pron', 'notes']].head(10))
else:
    print("  없음")

---

## 🔬 9️⃣ 형태소 특성과 ㄴ 삽입 상관관계

In [ ]:
# 빈도와 ㄴ 삽입 비율 상관관계
print("=" * 60)
print("형태소 빈도 vs ㄴ 삽입 비율 (morph1)")
print("=" * 60)

# morph1 형태소 빈도와 yes_ratio
df_morph1_freq = df_morph1_n_ins.copy()
df_morph1_freq = df_morph1_freq.sort_values('yes_ratio', ascending=False)

print("\nㄴ 삽입 비율이 높은 형태소 (80% 이상):")
high_n_ins = df_morph1_freq[df_morph1_freq['yes_ratio'] >= 0.8]
if len(high_n_ins) > 0:
    print(high_n_ins[['morph', 'total', 'yes_ratio']].to_string(index=False))
else:
    print("  없음")

print("\nㄴ 삽입 비율이 낮은 형태소 (30% 미만):")
low_n_ins = df_morph1_freq[df_morph1_freq['yes_ratio'] < 0.3]
if len(low_n_ins) > 0:
    print(low_n_ins[['morph', 'total', 'yes_ratio']].to_string(index=False))
else:
    print("  없음")

# 상관계수 계산 (출현 빈도 vs yes 비율)
if len(df_morph1_freq) >= 3:
    correlation = df_morph1_freq[['total', 'yes_ratio']].corr().iloc[0, 1]
    print(f"\n상관계수 (출현빈도 vs ㄴ삽입비율): {correlation:.3f}")
    if abs(correlation) < 0.3:
        print("  → 약한 상관관계 (빈도와 ㄴ 삽입은 독립적)")
    elif abs(correlation) < 0.7:
        print("  → 중간 상관관계")
    else:
        print("  → 강한 상관관계")

In [ ]:
# 품사와 ㄴ 삽입 비율
print("\n" + "=" * 60)
print("품사 (morph1_pos) vs ㄴ 삽입 비율")
print("=" * 60)

pos_n_insertion = df_keep.groupby('morph1_pos').agg({
    'n_insertion': lambda x: (x == 'yes').sum(),
    'word': 'count'
}).rename(columns={'n_insertion': 'n_yes', 'word': 'total'})

pos_n_insertion['yes_ratio'] = pos_n_insertion['n_yes'] / pos_n_insertion['total']
pos_n_insertion = pos_n_insertion.sort_values('yes_ratio', ascending=False)

print(pos_n_insertion)

---

## 💾 🔟 결과 저장

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# 1. 형태소 위치 통계
output1 = f'search_results/morpheme_positions_{timestamp}.csv'
df_positions.to_csv(output1, index=False, encoding='utf-8-sig')
print(f"✅ 형태소 위치 통계: {output1}")

# 2. 형태소 조합 통계
output2 = f'search_results/morpheme_combinations_{timestamp}.csv'
combinations.to_csv(output2, index=False, encoding='utf-8-sig')
print(f"✅ 형태소 조합 통계: {output2}")

# 3. 품사별 형태소
morph1_pos_counts = df_keep.groupby(['morph1', 'morph1_pos']).size().reset_index(name='count')
morph1_pos_counts = morph1_pos_counts.sort_values('count', ascending=False)
output3 = f'search_results/morph1_by_pos_{timestamp}.csv'
morph1_pos_counts.to_csv(output3, index=False, encoding='utf-8-sig')
print(f"✅ morph1 품사별 통계: {output3}")

# 4. morph1별 ㄴ 삽입 비율 ⭐
output4 = f'search_results/morph1_n_insertion_{timestamp}.csv'
df_morph1_n_ins.to_csv(output4, index=False, encoding='utf-8-sig')
print(f"✅ morph1 ㄴ삽입 비율: {output4}")

# 5. morph2별 ㄴ 삽입 비율 ⭐
output5 = f'search_results/morph2_n_insertion_{timestamp}.csv'
df_morph2_n_ins.to_csv(output5, index=False, encoding='utf-8-sig')
print(f"✅ morph2 ㄴ삽입 비율: {output5}")

# 6. 품사별 ㄴ 삽입 비율 ⭐
output6 = f'search_results/pos_n_insertion_{timestamp}.csv'
pos_n_insertion.to_csv(output6, encoding='utf-8-sig')
print(f"✅ 품사별 ㄴ삽입 비율: {output6}")

print("\n🎯 이 분석으로 알 수 있는 것:")
print("   - 어떤 형태소가 ㄴ 삽입을 '촉진'하는가?")
print("   - 형태소 빈도와 ㄴ 삽입의 관계")
print("   - 품사(접사/어근)와 ㄴ 삽입의 관계")
print("   - 사전 발음과 실제 발음의 차이")

---

## 📝 활용 방법

### 1. 생산적 형태소 파악
```
morph2 = "일" (10회)
→ "일"로 끝나는 합성명사가 많음
→ "밤일", "집일", "날일" 등
→ 생산적 패턴
```

### 2. 접사 식별
```
morph1에만 나오고 XPN (접두사)
→ 접두사

morph2에만 나오고 XSN (접미사)
→ 접미사
```

### 3. ㄴ 삽입 촉진 형태소 (⭐ 핵심!)
```
"솜" morph1일 때 → ㄴ 삽입 90%
"밤" morph1일 때 → ㄴ 삽입 80% (dialect)
"일" morph2일 때 → ㄴ 삽입 60%

→ 특정 형태소가 ㄴ 삽입을 "촉진"하는가?
→ 형태소 고유의 특성? vs 조합의 문제?
```

### 4. 논문 작성
```
Table 1: ㄴ 삽입 환경의 형태소 분포
- 가장 생산적인 선행 형태소: ...
- 가장 생산적인 후행 형태소: ...

Table 2: 형태소별 ㄴ 삽입 발생률 ⭐
- 고정적 ㄴ 삽입 형태소 (>80%): ...
- 변이적 ㄴ 삽입 형태소 (30-80%): ...
- ㄴ 삽입 억제 형태소 (<30%): ...
```